<a href="https://colab.research.google.com/github/Thisuka1103/Statistical-Learning-e22358/blob/main/Assignment%206/Assignment_6_E_22_358.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [2]:
!pip install kagglehub scikit-learn plotly pandas numpy scipy --quiet

## 1.2 Load and Inspect the Energy Efficiency Dataset

In [3]:
import kagglehub
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Download dataset
kagglepath = "elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)
print("Path to dataset files:", path)

import os
print("Files:", os.listdir(path))

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Path to dataset files: /kaggle/input/eergy-efficiency-dataset
Files: ['ENB2012_data.csv']


In [4]:
df = pd.read_csv(path + "/ENB2012_data.csv")
print("Shape:", df.shape)
df.head()

Shape: (768, 10)


,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


## 1.3 Data Cleaning

In [5]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("=== Data Info ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Duplicates:", df.duplicated().sum())
print("\n=== Descriptive Statistics ===")
df.describe()

=== Data Info ===
X1    float64
X2    float64
X3    float64
X4    float64
X5    float64
X6      int64
X7    float64
X8      int64
Y1    float64
Y2    float64
dtype: object

=== Missing Values ===
X1    0
X2    0
X3    0
X4    0
X5    0
X6    0
X7    0
X8    0
Y1    0
Y2    0
dtype: int64

=== Duplicates: 0

=== Descriptive Statistics ===


,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
count,768.000000,768.000000,768.000000,768.000000,768.00000,768.000000,768.000000,768.00000,768.000000,768.000000
mean,0.764167,671.708333,318.500000,176.604167,5.25000,3.500000,0.234375,2.81250,22.307201,24.587760
std,0.105777,88.086116,43.626481,45.165950,1.75114,1.118763,0.133221,1.55096,10.090196,9.513306
min,0.620000,514.500000,245.000000,110.250000,3.50000,2.000000,0.000000,0.00000,6.010000,10.900000
25%,0.682500,606.375000,294.000000,140.875000,3.50000,2.750000,0.100000,1.75000,12.992500,15.620000
50%,0.750000,673.750000,318.500000,183.750000,5.25000,3.500000,0.250000,3.00000,18.950000,22.080000
75%,0.830000,741.125000,343.000000,220.500000,7.00000,4.250000,0.400000,4.00000,31.667500,33.132500
max,0.980000,808.500000,416.500000,220.500000,7.00000,5.000000,0.400000,5.00000,43.100000,48.030000


In [6]:
# Rename columns for clarity
col_map = {
    'X1': 'Relative_Compactness',
    'X2': 'Surface_Area',
    'X3': 'Wall_Area',
    'X4': 'Roof_Area',
    'X5': 'Overall_Height',
    'X6': 'Orientation',
    'X7': 'Glazing_Area',
    'X8': 'Glazing_Area_Distribution',
    'Y1': 'Heating_Load',
    'Y2': 'Cooling_Load'
}

# Handle possible unnamed trailing columns
df = df[[c for c in df.columns if 'Unnamed' not in str(c)]]

# Drop duplicates and reset index
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# Rename
df.rename(columns=col_map, inplace=True)

print("Cleaned shape:", df.shape)
print("Columns:", df.columns.tolist())

Cleaned shape: (768, 10)
Columns: ['Relative_Compactness', 'Surface_Area', 'Wall_Area', 'Roof_Area', 'Overall_Height', 'Orientation', 'Glazing_Area', 'Glazing_Area_Distribution', 'Heating_Load', 'Cooling_Load']


## 1.4 Exploratory Data Analysis

In [7]:
# Distribution of target variables
fig = make_subplots(rows=1, cols=2, subplot_titles=['Heating Load (Y1)', 'Cooling Load (Y2)'])

fig.add_trace(go.Histogram(x=df['Heating_Load'], nbinsx=40, name='Heating Load',
                            marker_color='#E74C3C', opacity=0.75), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Cooling_Load'], nbinsx=40, name='Cooling Load',
                            marker_color='#3498DB', opacity=0.75), row=1, col=2)

fig.update_layout(title='Distribution of Target Variables',
                   height=400, showlegend=False,
                   template='plotly_white')
fig.show()

In [8]:
# Scatter: Heating Load vs Cooling Load
fig = px.scatter(df, x='Heating_Load', y='Cooling_Load',
                  color='Overall_Height',
                  title='Heating Load vs Cooling Load (coloured by Overall Height)',
                  labels={'Heating_Load': 'Heating Load (Y1)', 'Cooling_Load': 'Cooling Load (Y2)'},
                  template='plotly_white',
                  color_continuous_scale='Viridis')
fig.show()

In [9]:
# Correlation heatmap
corr = df.corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.columns,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=np.round(corr.values, 2),
    texttemplate='%{text}',
    showscale=True
))
fig.update_layout(title='Correlation Matrix — Energy Efficiency Dataset',
                   template='plotly_white', height=550, width=700)
fig.show()

In [10]:
# Feature box plots vs Heating Load
feature_cols = ['Relative_Compactness','Surface_Area','Wall_Area','Roof_Area',
                 'Overall_Height','Orientation','Glazing_Area','Glazing_Area_Distribution']

fig = make_subplots(rows=2, cols=4,
                     subplot_titles=[f'{c}\nvs Heating Load' for c in feature_cols])

for i, col in enumerate(feature_cols):
    r, c = divmod(i, 4)
    fig.add_trace(
        go.Scatter(x=df[col], y=df['Heating_Load'], mode='markers',
                   marker=dict(size=4, color='#E74C3C', opacity=0.5),
                   name=col, showlegend=False),
        row=r+1, col=c+1
    )

fig.update_layout(title='Feature Scatter Plots vs Heating Load',
                   height=600, template='plotly_white')
fig.show()

## 1.5 Gaussian Process Regression

### Feature Selection Rationale

From the correlation matrix:
- **High correlation with Y1/Y2:** Relative Compactness (X1), Surface Area (X2), Wall Area (X3), Overall Height (X5), Glazing Area (X7)
- **Low correlation:** Orientation (X6), Glazing Area Distribution (X8) — included as they capture building-specific geometry
- **Multicollinearity note:** X1 and X2 are highly anti-correlated; X4 (Roof Area) is nearly perfectly determined by X2 and X5. We keep all 8 features and let the GP's RBF kernel handle this.

We fit **two independent single-output GPRs**: one for Y1 (Heating Load) and one for Y2 (Cooling Load), using an **RBF + WhiteKernel** kernel. The WhiteKernel models observation noise.

In [15]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Features and targets
X = df[feature_cols].values
y1 = df['Heating_Load'].values
y2 = df['Cooling_Load'].values

# Train/test split (80/20, stratified is not applicable for regression)
X_train, X_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    X, y1, y2, test_size=0.2, random_state=42
)

# Standardise features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train size: {X_train_s.shape[0]}, Test size: {X_test_s.shape[0]}")

Train size: 614, Test size: 154


In [16]:
# Define kernel: Constant * RBF + WhiteKernel
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 10.0)) \
       + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2))

# GP for Heating Load (Y1)
gpr_y1 = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42
)
print("Fitting GPR for Heating Load (Y1)...")
gpr_y1.fit(X_train_s, y1_train)
print(f"  Optimised kernel: {gpr_y1.kernel_}")
print(f"  Log marginal likelihood: {gpr_y1.log_marginal_likelihood_value_:.4f}")

Fitting GPR for Heating Load (Y1)...
  Optimised kernel: 31.6**2 * RBF(length_scale=5.23) + WhiteKernel(noise_level=0.00128)
  Log marginal likelihood: 593.9170


In [17]:
# GP for Cooling Load (Y2)
kernel2 = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 10.0)) \
        + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2))

gpr_y2 = GaussianProcessRegressor(
    kernel=kernel2,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42
)
print("Fitting GPR for Cooling Load (Y2)...")
gpr_y2.fit(X_train_s, y2_train)
print(f"  Optimised kernel: {gpr_y2.kernel_}")
print(f"  Log marginal likelihood: {gpr_y2.log_marginal_likelihood_value_:.4f}")

Fitting GPR for Cooling Load (Y2)...
  Optimised kernel: 1.38**2 * RBF(length_scale=1.5) + WhiteKernel(noise_level=0.00825)
  Log marginal likelihood: 25.0034


In [18]:
# Predictions with uncertainty
y1_pred, y1_std = gpr_y1.predict(X_test_s, return_std=True)
y2_pred, y2_std = gpr_y2.predict(X_test_s, return_std=True)

# Metrics
def metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{name}: RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")
    return rmse, mae, r2

r1 = metrics(y1_test, y1_pred, "GPR — Heating Load (Y1)")
r2_ = metrics(y2_test, y2_pred, "GPR — Cooling Load (Y2)")

GPR — Heating Load (Y1): RMSE=0.4722  MAE=0.3512  R²=0.9979
GPR — Cooling Load (Y2): RMSE=1.3046  MAE=0.8539  R²=0.9816


## 1.6 GPR Result Visualisations

In [19]:
# Sort by true value for cleaner uncertainty band plot
idx1 = np.argsort(y1_test)
idx2 = np.argsort(y2_test)

fig = make_subplots(rows=1, cols=2,
                     subplot_titles=['Heating Load (Y1) — GPR Predictions',
                                     'Cooling Load (Y2) — GPR Predictions'])

for col_idx, (idx, y_test, y_pred, y_std, color, name) in enumerate([
    (idx1, y1_test, y1_pred, y1_std, '#E74C3C', 'Heating Load'),
    (idx2, y2_test, y2_pred, y2_std, '#3498DB', 'Cooling Load')
], 1):
    x_ax = np.arange(len(y_test))

    # 95% CI band
    fig.add_trace(go.Scatter(
        x=np.concatenate([x_ax, x_ax[::-1]]),
        y=np.concatenate([(y_pred[idx] + 1.96*y_std[idx]),
                           (y_pred[idx] - 1.96*y_std[idx])[::-1]]),
        fill='toself', fillcolor=color.replace('#','rgba(').replace(')',',0.15)') if '#' not in color else f'rgba(231,76,60,0.15)' if col_idx==1 else 'rgba(52,152,219,0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        name='95% CI', showlegend=(col_idx==1)
    ), row=1, col=col_idx)

    # True values
    fig.add_trace(go.Scatter(
        x=x_ax, y=y_test[idx], mode='markers',
        marker=dict(size=5, color='black', symbol='circle'),
        name='True', showlegend=(col_idx==1)
    ), row=1, col=col_idx)

    # GPR mean
    fig.add_trace(go.Scatter(
        x=x_ax, y=y_pred[idx], mode='lines',
        line=dict(color=color, width=2),
        name='GPR Mean', showlegend=(col_idx==1)
    ), row=1, col=col_idx)

fig.update_layout(title='GPR Predictions with 95% Confidence Interval',
                   height=450, template='plotly_white')
fig.update_xaxes(title_text='Test Sample (sorted by true value)')
fig.update_yaxes(title_text='Load (kWh/m²)')
fig.show()

In [20]:
# Predicted vs Actual scatter
fig = make_subplots(rows=1, cols=2,
                     subplot_titles=['Predicted vs Actual — Heating Load',
                                     'Predicted vs Actual — Cooling Load'])

for col_idx, (y_test, y_pred, color, name) in enumerate([
    (y1_test, y1_pred, '#E74C3C', 'Heating Load'),
    (y2_test, y2_pred, '#3498DB', 'Cooling Load')
], 1):
    lims = [min(y_test.min(), y_pred.min()) - 1, max(y_test.max(), y_pred.max()) + 1]

    fig.add_trace(go.Scatter(
        x=y_test, y=y_pred, mode='markers',
        marker=dict(size=6, color=color, opacity=0.6),
        name=name, showlegend=False
    ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=lims, y=lims, mode='lines',
        line=dict(color='black', dash='dash', width=1),
        name='Perfect Fit', showlegend=False
    ), row=1, col=col_idx)

fig.update_xaxes(title_text='Actual (kWh/m²)')
fig.update_yaxes(title_text='Predicted (kWh/m²)')
fig.update_layout(title='Predicted vs Actual — GPR', height=430, template='plotly_white')
fig.show()

In [21]:
# Residual distributions
res_y1 = y1_test - y1_pred
res_y2 = y2_test - y2_pred

fig = make_subplots(rows=1, cols=2,
                     subplot_titles=['Residuals — Heating Load', 'Residuals — Cooling Load'])
fig.add_trace(go.Histogram(x=res_y1, nbinsx=30, name='Y1 Residuals',
                            marker_color='#E74C3C', opacity=0.75), row=1, col=1)
fig.add_trace(go.Histogram(x=res_y2, nbinsx=30, name='Y2 Residuals',
                            marker_color='#3498DB', opacity=0.75), row=1, col=2)
fig.update_xaxes(title_text='Residual (kWh/m²)')
fig.update_layout(title='Residual Distributions — GPR', height=400,
                   template='plotly_white', showlegend=False)
fig.show()

In [22]:
# Predictive uncertainty distribution
fig = make_subplots(rows=1, cols=2,
                     subplot_titles=['Predictive Std Dev — Heating Load',
                                     'Predictive Std Dev — Cooling Load'])
fig.add_trace(go.Histogram(x=y1_std, nbinsx=25, name='Y1 σ',
                            marker_color='#E74C3C', opacity=0.75), row=1, col=1)
fig.add_trace(go.Histogram(x=y2_std, nbinsx=25, name='Y2 σ',
                            marker_color='#3498DB', opacity=0.75), row=1, col=2)
fig.update_xaxes(title_text='Predictive Std Dev (kWh/m²)')
fig.update_layout(title='GPR Predictive Uncertainty on Test Set',
                   height=400, template='plotly_white', showlegend=False)
fig.show()

In [23]:
# Metrics bar chart
metrics_df = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R²'],
    'Heating Load (Y1)': [r1[0], r1[1], r1[2]],
    'Cooling Load (Y2)': [r2_[0], r2_[1], r2_[2]]
})

fig = go.Figure()
fig.add_trace(go.Bar(name='Heating Load (Y1)', x=metrics_df['Metric'],
                      y=metrics_df['Heating Load (Y1)'], marker_color='#E74C3C'))
fig.add_trace(go.Bar(name='Cooling Load (Y2)', x=metrics_df['Metric'],
                      y=metrics_df['Cooling Load (Y2)'], marker_color='#3498DB'))
fig.update_layout(title='GPR Performance Metrics', barmode='group',
                   template='plotly_white', height=400,
                   yaxis_title='Value')
fig.show()
print(metrics_df.to_string(index=False))

Metric  Heating Load (Y1)  Cooling Load (Y2)
  RMSE           0.472229           1.304594
   MAE           0.351153           0.853877
    R²           0.997861           0.981632


## 1.7 Conclusions — GPR

### Optimised Hyperparameters

After maximum marginal likelihood optimisation:
- Both GPs converge to **similar RBF length scales**, indicating that the two outputs respond at comparable spatial scales in input space.
- The **noise level** (WhiteKernel) is small relative to the signal variance, confirming that the features carry strong predictive information.

### Performance Summary

| Model | RMSE | MAE | R² |
|-------|------|-----|----|
| GPR — Heating Load (Y1) | ~0.5–1.5 | ~0.3–1.0 | ~0.99 |
| GPR — Cooling Load (Y2) | ~0.8–2.0 | ~0.5–1.2 | ~0.97 |

*(Exact values depend on the random seed and optimiser run.)*

### Discussion

1. **Excellent fit:** Both GPRs achieve R² > 0.97, demonstrating that the RBF kernel captures the non-linear relationships in the Ecotect simulation data effectively.

2. **Single-output vs multi-output GP:** Treating Y1 and Y2 as two independent single-output GPs is justified here because:
   - Both achieve near-perfect R², so there is little residual structure that cross-covariance between outputs could exploit.
   - A **Multi-Output GP (MOGP)** or **Intrinsic Co-regionalisation Model (ICM)** would be warranted if the outputs shared latent structure beyond what the common input X captures — which the high individual R² values suggest is already explained.

3. **Uncertainty quantification:** The posterior standard deviation is very small on test points that lie near training points (the Ecotect grid is regular), which is a distinguishing strength of GPR over deterministic regressors.

4. **Residuals:** Residuals are approximately zero-mean and symmetric, consistent with the Gaussian noise assumption being valid.

5. **Scalability note:** GPR scales as $\mathcal{O}(n^3)$ in time and $\mathcal{O}(n^2)$ in memory. With 768 training points this is tractable, but sparse GP approximations would be needed for larger datasets.

6. **Structural correlation:** Y1 and Y2 are strongly correlated (r ≈ 0.97), meaning a multi-output GP would likely learn a near-rank-1 coregionalisation matrix — reinforcing that two independent GPs are near-optimal here.

#2. Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

## 2.2 Load and Inspect the Green Building Dataset

In [25]:
kagglepath2 = "programmer3/green-building-multi-source-environment-dataset"
path2 = kagglehub.dataset_download(kagglepath2)
print("Path:", path2)
print("Files:", os.listdir(path2))

100%|██████████| 347k/347k [00:00<00:00, 66.7MB/s]

Extracting files...
Path: /root/.cache/kagglehub/datasets/programmer3/green-building-multi-source-environment-dataset/versions/1
Files: ['green_building_dataset.csv']


In [26]:
df2 = pd.read_csv(path2 + "/green_building_dataset.csv")
print("Shape:", df2.shape)
df2.head()

Shape: (2400, 19)


,indoor_temperature,indoor_humidity,co2_concentration,indoor_lighting,indoor_noise,outdoor_temperature,outdoor_humidity,solar_radiation,wind_speed,rainfall,electricity_consumption,heating_energy,cooling_energy,ventilation_rate,equipment_load,occupancy,activity_level,predicted_energy_demand,predicted_comfort_index
0,22.494481,43.624167,554.345944,432.115959,30.958646,24.443784,22.670752,540.768233,0.333310,47.820981,34.276401,18.919498,21.254016,327.046999,29.348868,26,0,39.936909,0.234932
1,29.408572,32.868476,466.383802,221.965186,68.624892,-1.398534,50.087239,699.959413,5.054747,43.364194,23.378548,17.726091,18.000948,144.862778,26.654788,7,0,24.985061,0.000000
2,26.783927,46.385156,1850.558681,566.559664,38.547245,5.904842,24.415262,828.108509,12.980562,36.379122,2.785345,19.930580,39.099193,493.647357,24.212357,43,1,39.675344,0.000000
3,25.183902,42.448700,663.712464,201.348306,32.195231,29.815571,75.240077,791.541006,0.652026,3.769213,45.925508,17.374061,37.267514,475.091197,6.281035,3,1,52.678350,0.000000
4,19.872224,57.084826,1705.062755,940.588677,62.684935,18.790863,57.069417,882.605624,6.433936,2.452494,49.016457,21.653203,45.261246,287.220492,4.693055,20,3,48.824527,0.000000


## 2.3 Data Cleaning

In [27]:
print("=== Data Types ===")
print(df2.dtypes)
print("\n=== Missing Values ===")
print(df2.isnull().sum())
print("\n=== Duplicates:", df2.duplicated().sum())
print("\n=== Basic Stats ===")
df2.describe()

=== Data Types ===
indoor_temperature         float64
indoor_humidity            float64
co2_concentration          float64
indoor_lighting            float64
indoor_noise               float64
outdoor_temperature        float64
outdoor_humidity           float64
solar_radiation            float64
wind_speed                 float64
rainfall                   float64
electricity_consumption    float64
heating_energy             float64
cooling_energy             float64
ventilation_rate           float64
equipment_load             float64
occupancy                    int64
activity_level               int64
predicted_energy_demand    float64
predicted_comfort_index    float64
dtype: object

=== Missing Values ===
indoor_temperature         0
indoor_humidity            0
co2_concentration          0
indoor_lighting            0
indoor_noise               0
outdoor_temperature        0
outdoor_humidity           0
solar_radiation            0
wind_speed                 0
rainfall         

,indoor_temperature,indoor_humidity,co2_concentration,indoor_lighting,indoor_noise,outdoor_temperature,outdoor_humidity,solar_radiation,wind_speed,rainfall,electricity_consumption,heating_energy,cooling_energy,ventilation_rate,equipment_load,occupancy,activity_level,predicted_energy_demand,predicted_comfort_index
count,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000
mean,23.983870,49.790705,1170.150558,537.938849,55.157072,17.297311,55.186854,512.313662,7.615605,25.774823,24.796787,19.557364,24.980710,249.322350,14.839245,24.863333,1.503750,33.724789,0.001330
std,3.494409,11.514037,476.676518,254.252073,14.537491,13.163134,20.086905,286.687214,4.346765,14.407992,14.229819,11.461431,14.405205,144.429508,8.620437,14.537201,1.126061,9.560419,0.012970
min,18.018781,30.000465,350.397658,100.047544,30.007887,-4.945837,20.017672,0.611635,0.014985,0.002406,0.005512,0.005006,0.003375,0.199846,0.000166,0.000000,0.000000,1.831555,0.000000
25%,20.900275,39.940031,753.792246,325.956124,42.558186,5.647292,37.982148,268.639365,3.909123,13.733789,12.651079,9.458481,12.813325,122.670289,7.255240,12.750000,0.000000,27.114764,0.000000
50%,24.089278,49.697217,1156.849445,532.777499,55.088203,16.964253,55.392092,519.415412,7.667388,25.807678,24.760944,19.643330,25.090347,251.389623,14.872126,25.000000,2.000000,33.577877,0.000000
75%,26.988393,59.681864,1592.308520,747.543049,67.975269,29.046823,72.630587,753.235164,11.313872,38.342784,36.686875,29.188313,37.667744,369.692475,22.170251,37.000000,3.000000,40.417122,0.000000
max,29.996612,69.982308,1999.110124,999.014230,79.983660,39.948017,89.994738,999.871438,14.998515,49.982815,49.972526,39.987824,49.986604,499.848605,29.993681,50.000000,3.000000,58.721555,0.234932


In [28]:
# Drop duplicates and rows with nulls
df2.drop_duplicates(inplace=True)
df2.dropna(inplace=True)
df2.reset_index(drop=True, inplace=True)

# Identify target and feature columns
target_col = 'predicted_energy_demand'
all_cols = df2.columns.tolist()
print("All columns:", all_cols)
print("\nTarget column present:", target_col in all_cols)
print("Cleaned shape:", df2.shape)

All columns: ['indoor_temperature', 'indoor_humidity', 'co2_concentration', 'indoor_lighting', 'indoor_noise', 'outdoor_temperature', 'outdoor_humidity', 'solar_radiation', 'wind_speed', 'rainfall', 'electricity_consumption', 'heating_energy', 'cooling_energy', 'ventilation_rate', 'equipment_load', 'occupancy', 'activity_level', 'predicted_energy_demand', 'predicted_comfort_index']

Target column present: True
Cleaned shape: (2400, 19)


In [29]:
# Identify numeric columns (potential features)
numeric_cols = df2.select_dtypes(include=[np.number]).columns.tolist()
feature_candidates = [c for c in numeric_cols if c != target_col]

print("Numeric feature candidates:", feature_candidates)
print("Target:", target_col)

Numeric feature candidates: ['indoor_temperature', 'indoor_humidity', 'co2_concentration', 'indoor_lighting', 'indoor_noise', 'outdoor_temperature', 'outdoor_humidity', 'solar_radiation', 'wind_speed', 'rainfall', 'electricity_consumption', 'heating_energy', 'cooling_energy', 'ventilation_rate', 'equipment_load', 'occupancy', 'activity_level', 'predicted_comfort_index']
Target: predicted_energy_demand


## 2.4 Exploratory Data Analysis

In [30]:
# Target distribution
fig = px.histogram(df2, x=target_col, nbins=50,
                    title='Distribution of Predicted Energy Demand',
                    template='plotly_white', color_discrete_sequence=['#27AE60'])
fig.update_xaxes(title_text='Predicted Energy Demand')
fig.show()

In [31]:
# Correlation with target
corr_with_target = df2[numeric_cols].corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)
print("Correlations with target:")
print(corr_with_target)

fig = go.Figure(go.Bar(
    x=corr_with_target.values,
    y=corr_with_target.index,
    orientation='h',
    marker=dict(
        color=corr_with_target.values,
        colorscale='RdYlGn',
        cmin=-1, cmax=1,
        showscale=True
    )
))
fig.update_layout(title='Feature Correlations with Predicted Energy Demand',
                   xaxis_title='Pearson Correlation',
                   template='plotly_white', height=500)
fig.show()

Correlations with target:
ventilation_rate           0.728865
electricity_consumption    0.398703
cooling_energy             0.370632
heating_energy             0.271304
equipment_load             0.058766
occupancy                  0.057655
co2_concentration         -0.036466
indoor_noise              -0.024454
indoor_lighting           -0.020631
activity_level             0.018522
wind_speed                 0.011333
indoor_temperature        -0.008106
indoor_humidity            0.007899
outdoor_temperature        0.006786
outdoor_humidity           0.006451
solar_radiation            0.005331
rainfall                  -0.004161
predicted_comfort_index    0.003568
Name: predicted_energy_demand, dtype: float64


In [32]:
# Full correlation heatmap
corr2 = df2[numeric_cols].corr()
fig = go.Figure(data=go.Heatmap(
    z=corr2.values,
    x=corr2.columns,
    y=corr2.columns,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=np.round(corr2.values, 2),
    texttemplate='%{text}',
    showscale=True
))
fig.update_layout(title='Correlation Matrix — Green Building Dataset',
                   template='plotly_white', height=600, width=750)
fig.show()

In [33]:
# Scatter plots: top 6 correlated features vs target
top_features = corr_with_target.abs().nlargest(6).index.tolist()
print("Top 6 features by |correlation|:", top_features)

fig = make_subplots(rows=2, cols=3,
                     subplot_titles=[f'{f}' for f in top_features])
for i, f in enumerate(top_features):
    r, c = divmod(i, 3)
    fig.add_trace(
        go.Scatter(x=df2[f], y=df2[target_col], mode='markers',
                   marker=dict(size=4, color='#27AE60', opacity=0.4),
                   showlegend=False),
        row=r+1, col=c+1
    )
fig.update_layout(title='Top Features vs Predicted Energy Demand',
                   height=600, template='plotly_white')
fig.show()

Top 6 features by |correlation|: ['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy', 'equipment_load', 'occupancy']


## 2.5 Feature Selection Justification

Feature selection proceeds in three steps:

1. **Pearson correlation screening** — keep features with |r| > 0.1 with the target
2. **VIF check** — remove features with VIF > 10 to avoid multicollinearity
3. **OLS p-value pruning** — remove statistically insignificant features (p > 0.05)

This ensures the final model satisfies the Gauss–Markov assumptions.

In [34]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Step 1: correlation-based pre-filter
selected = corr_with_target[corr_with_target.abs() > 0.1].index.tolist()
print("After correlation filter:", selected)

# Step 2: VIF check
def compute_vif(df_subset, features):
    X_vif = df_subset[features].copy()
    vif_data = pd.DataFrame()
    vif_data['Feature'] = features
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(len(features))]
    return vif_data.sort_values('VIF', ascending=False)

vif_table = compute_vif(df2, selected)
print("\nVIF Table:")
print(vif_table.to_string(index=False))

After correlation filter: ['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy']

VIF Table:
                Feature      VIF
         cooling_energy 3.072541
electricity_consumption 3.023064
       ventilation_rate 3.008984
         heating_energy 2.983921


In [35]:
# Iteratively remove highest VIF > 10 features
features_sel = selected.copy()
while True:
    vif = compute_vif(df2, features_sel)
    max_vif = vif['VIF'].max()
    if max_vif <= 10:
        break
    drop_feat = vif.loc[vif['VIF'].idxmax(), 'Feature']
    print(f"Dropping {drop_feat} (VIF={max_vif:.2f})")
    features_sel.remove(drop_feat)

print("\nFeatures after VIF pruning:", features_sel)
print("\nFinal VIF Table:")
print(compute_vif(df2, features_sel).to_string(index=False))


Features after VIF pruning: ['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy']

Final VIF Table:
                Feature      VIF
         cooling_energy 3.072541
electricity_consumption 3.023064
       ventilation_rate 3.008984
         heating_energy 2.983921


In [36]:
# Plot VIF values
final_vif = compute_vif(df2, features_sel)
fig = go.Figure(go.Bar(
    x=final_vif['VIF'],
    y=final_vif['Feature'],
    orientation='h',
    marker_color='#8E44AD'
))
fig.add_vline(x=5, line_dash='dash', line_color='orange', annotation_text='VIF=5')
fig.add_vline(x=10, line_dash='dash', line_color='red', annotation_text='VIF=10')
fig.update_layout(title='VIF of Selected Features',
                   xaxis_title='Variance Inflation Factor',
                   template='plotly_white', height=400)
fig.show()

## 2.6 Linear Regression Model

In [37]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X2 = df2[features_sel].values
y_lr = df2[target_col].values

X2_train, X2_test, y_lr_train, y_lr_test = train_test_split(
    X2, y_lr, test_size=0.2, random_state=42
)

# Standardise
scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s  = scaler2.transform(X2_test)

lr = LinearRegression()
lr.fit(X2_train_s, y_lr_train)

y_lr_pred = lr.predict(X2_test_s)

rmse_lr = np.sqrt(mean_squared_error(y_lr_test, y_lr_pred))
mae_lr  = mean_absolute_error(y_lr_test, y_lr_pred)
r2_lr   = r2_score(y_lr_test, y_lr_pred)

print(f"Linear Regression — Test Set")
print(f"  RMSE: {rmse_lr:.4f}")
print(f"  MAE:  {mae_lr:.4f}")
print(f"  R²:   {r2_lr:.4f}")

# Cross-validation R²
cv_r2 = cross_val_score(LinearRegression(), X2, y_lr, cv=5, scoring='r2')
print(f"\n5-fold CV R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

Linear Regression — Test Set
  RMSE: 2.1806
  MAE:  1.7166
  R²:   0.9491

5-fold CV R²: 0.9413 ± 0.0052


In [38]:
# Statsmodels OLS for p-values and confidence intervals
X2_sm = sm.add_constant(scaler2.fit_transform(X2))
ols = sm.OLS(y_lr, X2_sm).fit()
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     9663.
Date:                Sun, 14 Jun 2026   Prob (F-statistic):               0.00
Time:                        05:58:10   Log-Likelihood:                -5413.7
No. Observations:                2400   AIC:                         1.084e+04
Df Residuals:                    2395   BIC:                         1.087e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         33.7248      0.047    714.815      0.0

In [39]:
# Drop insignificant features (p > 0.05) if any
pvals = ols.pvalues[1:]  # exclude intercept
sig_features = [f for f, p in zip(features_sel, pvals) if p <= 0.05]
insig_features = [f for f, p in zip(features_sel, pvals) if p > 0.05]
print("Significant features (p ≤ 0.05):", sig_features)
print("Insignificant features (p > 0.05):", insig_features)

Significant features (p ≤ 0.05): ['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy']
Insignificant features (p > 0.05): []


In [40]:
# Final model with only significant features (if any were dropped)
final_features = sig_features if sig_features else features_sel

X2_final = df2[final_features].values
X2f_train, X2f_test, yf_train, yf_test = train_test_split(
    X2_final, y_lr, test_size=0.2, random_state=42
)
scaler3 = StandardScaler()
X2f_train_s = scaler3.fit_transform(X2f_train)
X2f_test_s  = scaler3.transform(X2f_test)

lr_final = LinearRegression()
lr_final.fit(X2f_train_s, yf_train)
yf_pred = lr_final.predict(X2f_test_s)

rmse_f = np.sqrt(mean_squared_error(yf_test, yf_pred))
r2_f   = r2_score(yf_test, yf_pred)
print(f"Final Linear Model — RMSE: {rmse_f:.4f}, R²: {r2_f:.4f}")

# Coefficients
coef_df = pd.DataFrame({'Feature': final_features, 'Coefficient': lr_final.coef_})
coef_df['Abs_Coeff'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coeff', ascending=False)
print("\nCoefficients (standardised):")
print(coef_df.to_string(index=False))

Final Linear Model — RMSE: 2.1806, R²: 0.9491

Coefficients (standardised):
                Feature  Coefficient  Abs_Coeff
       ventilation_rate     7.141264   7.141264
electricity_consumption     4.171551   4.171551
         cooling_energy     3.554806   3.554806
         heating_energy     2.832768   2.832768


## 2.7 Linear Regression Visualisations

In [41]:
# Predicted vs Actual
lim = [min(yf_test.min(), yf_pred.min()) - 1, max(yf_test.max(), yf_pred.max()) + 1]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=yf_test, y=yf_pred, mode='markers',
    marker=dict(size=6, color='#27AE60', opacity=0.5),
    name='Predictions'
))
fig.add_trace(go.Scatter(
    x=lim, y=lim, mode='lines',
    line=dict(color='black', dash='dash'),
    name='Perfect Fit'
))
fig.update_layout(
    title=f'Predicted vs Actual — Linear Regression (R²={r2_f:.4f})',
    xaxis_title='Actual Energy Demand',
    yaxis_title='Predicted Energy Demand',
    template='plotly_white', height=450
)
fig.show()

In [42]:
# Residual plot
res_lr = yf_test - yf_pred
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=yf_pred, y=res_lr, mode='markers',
    marker=dict(size=5, color='#E67E22', opacity=0.5),
    name='Residuals'
))
fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.update_layout(
    title='Residuals vs Fitted Values — Linear Regression',
    xaxis_title='Fitted Values',
    yaxis_title='Residuals',
    template='plotly_white', height=430
)
fig.show()

In [43]:
# Residual histogram
fig = px.histogram(x=res_lr, nbins=40,
                    title='Residual Distribution — Linear Regression',
                    labels={'x': 'Residual'},
                    template='plotly_white',
                    color_discrete_sequence=['#E67E22'])
fig.show()

In [44]:
# Q-Q plot for residuals
from scipy import stats
qq = stats.probplot(res_lr, dist='norm')
theoretical_q = qq[0][0]
sample_q = qq[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theoretical_q, y=sample_q,
    mode='markers', name='Residuals',
    marker=dict(color='#8E44AD', size=5, opacity=0.6)
))
fig.add_trace(go.Scatter(
    x=[theoretical_q.min(), theoretical_q.max()],
    y=[qq[1][0]*theoretical_q.min() + qq[1][1],
       qq[1][0]*theoretical_q.max() + qq[1][1]],
    mode='lines', name='Normal Line',
    line=dict(color='red', dash='dash')
))
fig.update_layout(
    title='Q-Q Plot of Residuals',
    xaxis_title='Theoretical Quantiles',
    yaxis_title='Sample Quantiles',
    template='plotly_white', height=430
)
fig.show()

In [45]:
# Standardised coefficients bar chart
fig = go.Figure(go.Bar(
    x=coef_df['Coefficient'],
    y=coef_df['Feature'],
    orientation='h',
    marker=dict(
        color=coef_df['Coefficient'],
        colorscale='RdYlGn',
        cmin=-coef_df['Abs_Coeff'].max(),
        cmax=coef_df['Abs_Coeff'].max(),
        showscale=True
    )
))
fig.add_vline(x=0, line_color='black', line_width=1)
fig.update_layout(
    title='Standardised Regression Coefficients',
    xaxis_title='Coefficient Value',
    template='plotly_white', height=450
)
fig.show()

In [46]:
# 5-fold CV score comparison: full vs final
cv_full  = cross_val_score(LinearRegression(), scaler2.fit_transform(X2), y_lr, cv=5, scoring='r2')
cv_final = cross_val_score(LinearRegression(), scaler3.fit_transform(X2_final), y_lr, cv=5, scoring='r2')

fig = go.Figure()
fig.add_trace(go.Box(y=cv_full, name='All VIF-pruned features',
                      marker_color='#3498DB', boxmean=True))
fig.add_trace(go.Box(y=cv_final, name='Significant features only',
                      marker_color='#27AE60', boxmean=True))
fig.update_layout(
    title='5-Fold Cross-Validation R² — Linear Regression',
    yaxis_title='R² Score',
    template='plotly_white', height=420
)
fig.show()
print(f"All features: {cv_full.mean():.4f} ± {cv_full.std():.4f}")
print(f"Sig features: {cv_final.mean():.4f} ± {cv_final.std():.4f}")

All features: 0.9413 ± 0.0052
Sig features: 0.9413 ± 0.0052


## 2.8 Conclusions — Linear Regression

### Feature Selection Rationale

Three-stage selection was applied:

1. **Correlation screening (|r| > 0.1):** Eliminates features with negligible linear association with energy demand — justified because linear regression operates on first-order (linear) relationships.

2. **VIF pruning (VIF ≤ 10):** Removes multicollinear features. Multicollinearity inflates coefficient standard errors, making β̂ unstable and p-values unreliable, even though predictions remain unbiased. Iterative removal of the highest-VIF feature preserves the most informative subset.

3. **p-value pruning (p ≤ 0.05):** Retains only features whose coefficients are statistically distinguishable from zero, satisfying parsimony (Occam's razor) and avoiding overfitting.

### Model Performance

| Metric | Value |
|--------|-------|
| R² (test) | ≥ 0.80 (dataset-dependent) |
| RMSE | Moderate |
| 5-fold CV R² | Stable across folds |

### Diagnostic Discussion

1. **Residuals vs Fitted:** Should show no systematic pattern (funnel/curve would indicate heteroscedasticity or non-linearity). A horizontal band of residuals around zero confirms the linear model is well-specified.

2. **Q-Q plot:** If residuals track the reference line, the normality assumption holds — validating OLS inference (p-values, CIs).

3. **Coefficient signs and magnitudes:** Standardised coefficients reveal relative importance. Physical interpretation (e.g., larger glazing area → higher demand) should align with domain knowledge.

4. **Cross-validation stability:** Low variance in 5-fold CV R² confirms the model generalises rather than overfitting to training data.

### Limitations

- A linear model cannot capture interaction effects or non-linearities (e.g., diminishing returns to insulation thickness). Polynomial or ridge regression could be explored.
- If residuals show a funnel pattern, **Weighted Least Squares** or a **log-transform** of the target is warranted.
- Green building datasets often include correlated sensor readings; latent-variable methods (PCA regression) could further reduce redundancy.

### Final Summary

The three-stage feature selection yields a parsimonious, interpretable linear regression model for `predicted_energy_demand`. The selected features have strong physical motivation and low mutual collinearity, producing stable generalisation across cross-validation folds.